# P1-E16.7D-2 renamed (P1-E11.D_2) FINAL — Matched Official-Test Comparison of Frozen Conventional and Selected Models

## Purpose

This experiment directly resolves the final comparator ambiguity by placing the authenticated frozen conventional benchmark from P1-E16.7A and the frozen endpoint-selected CEEDNet/SSL models from P1-E16.7D on the **same CAUEEG official-test recordings**, endpoints, labels, and metric implementation.

No model is retrained, no EEG signal is reloaded, and no threshold is changed. The notebook consumes only previously frozen predictions and authenticated model records. It therefore adds a matched statistical comparison without creating another test-access cycle.

The evidence remains a **retrospective locked-test comparison**. Results are not used to retune or replace the P1-E16.7C selections. Normal-versus-MCI is cross-sectional discrimination and is not conversion prediction.

## 1. Configuration and exact upstream evidence


In [ ]:
from pathlib import Path
import hashlib, json, platform, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)

SEED=20261010
N_BOOT=10000
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
METHOD_CONVENTIONAL='Frozen conventional benchmark'
METHOD_SELECTED='Endpoint-selected CEEDNet/SSL'

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
E16_7A_ROOT=OUTPUTS_ROOT/'p1_e16_7a_four_endpoint_nested_conventional_freeze'
E16_7D_ROOT=OUTPUTS_ROOT/'p1_e16_7d_full_development_refit_retrospective_locked_test'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7d_2_matched_official_test_comparison'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'
for p in (TABLES,FIGURES): p.mkdir(parents=True,exist_ok=True)

CONV_PRED=E16_7A_ROOT/'tables'/'11_RETROSPECTIVE_LOCKED_TEST_PREDICTIONS.csv'
CONV_MODELS=E16_7A_ROOT/'tables'/'10_FROZEN_ENDPOINT_MODELS.csv'
CONV_INTEGRITY=E16_7A_ROOT/'tables'/'15_FINAL_INTEGRITY_CHECKS.csv'
CONV_RUN=E16_7A_ROOT/'run_record.json'
SELECTED_PRED=E16_7D_ROOT/'tables'/'04_RETROSPECTIVE_LOCKED_TEST_PREDICTIONS.csv'
SELECTED_MODELS=E16_7D_ROOT/'tables'/'05_SERIALIZED_MODEL_ARTIFACTS.csv'
SELECTED_INTEGRITY=E16_7D_ROOT/'tables'/'09_FINAL_INTEGRITY_CHECKS.csv'
SELECTED_RUN=E16_7D_ROOT/'run_record.json'

def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def json_default(x):
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,Path): return str(x)
    raise TypeError(type(x).__name__)
def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); out=np.empty(len(p),float); running=0.0
    for rank,idx in enumerate(order):
        running=max(running,(len(p)-rank)*p[idx]); out[idx]=min(1.0,running)
    return out
print({'python':sys.version.split()[0],'platform':platform.platform(),'output_root':str(OUTPUT_ROOT)})

## 2. Authenticate both frozen evidence streams


In [ ]:
required=[CONV_PRED,CONV_MODELS,CONV_INTEGRITY,CONV_RUN,SELECTED_PRED,SELECTED_MODELS,SELECTED_INTEGRITY,SELECTED_RUN]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Missing required frozen artifact(s):\n'+'\n'.join(missing))

def integrity_passed(path):
    q=pd.read_csv(path)
    if 'pass' not in q: return False
    return q['pass'].astype(str).str.lower().eq('true').all()
if not integrity_passed(CONV_INTEGRITY): raise AssertionError('P1-E16.7A integrity failure.')
if not integrity_passed(SELECTED_INTEGRITY): raise AssertionError('P1-E16.7D integrity failure.')
run_a=json.loads(CONV_RUN.read_text(encoding='utf-8')); run_d=json.loads(SELECTED_RUN.read_text(encoding='utf-8'))
if 'PASSED' not in str(run_a.get('status','')) or 'PASSED' not in str(run_d.get('status','')): raise AssertionError('An upstream run record is not PASSED.')
if run_d.get('post_test_tuning') not in (False,None): raise AssertionError('P1-E16.7D reports post-test tuning.')

conv_models=pd.read_csv(CONV_MODELS); selected_models=pd.read_csv(SELECTED_MODELS)
for r in conv_models.itertuples():
    if not Path(r.model_path).is_file() or sha256_file(Path(r.model_path))!=r.model_sha256: raise AssertionError(f'Conventional model authentication failed: {r.endpoint}')
for r in selected_models.itertuples():
    if not Path(r.model_path).is_file() or sha256_file(Path(r.model_path))!=r.model_sha256: raise AssertionError(f'Selected model authentication failed: {r.task}')

inputs=pd.DataFrame([{'role':p.name,'path':str(p),'sha256':sha256_file(p)} for p in required])
save_table(inputs,'01_AUTHENTICATED_INPUTS'); display(inputs)

## 3. Standardize predictions and prove recording-level comparability


In [ ]:
def standardize(path,method):
    q=pd.read_csv(path,dtype={'recording_id':str}).rename(columns={'endpoint':'task','fold':'outer_fold'})
    q['recording_id']=q.recording_id.astype(str).str.strip().str.zfill(5); q['comparison_method']=method
    needed={'task','recording_id','true_label','predicted_label'}
    if not needed.issubset(q): raise KeyError(f'{path.name} lacks {needed-set(q)}')
    return q
conv=standardize(CONV_PRED,METHOD_CONVENTIONAL)
selected=standardize(SELECTED_PRED,METHOD_SELECTED)

alignment=[]
for task,classes in TASKS.items():
    a=conv.loc[conv.task.eq(task)].set_index('recording_id').sort_index()
    b=selected.loc[selected.task.eq(task)].set_index('recording_id').sort_index()
    alignment.append({'task':task,'n_conventional':len(a),'n_selected':len(b),'same_recordings':a.index.equals(b.index),
                      'same_true_labels':a.index.equals(b.index) and a.true_label.equals(b.true_label),
                      'conventional_unique':not a.index.duplicated().any(),'selected_unique':not b.index.duplicated().any(),
                      'required_classes_present':set(a.true_label)==set(classes) and set(b.true_label)==set(classes)})
alignment=pd.DataFrame(alignment)
if not alignment.drop(columns=['task','n_conventional','n_selected']).all().all():
    display(alignment); raise AssertionError('The two official-test evidence streams are not exactly comparable.')
combined=pd.concat([conv,selected],ignore_index=True,sort=False)
save_table(alignment,'02_MATCHED_RECORDING_AND_LABEL_AUDIT'); save_table(combined,'03_MATCHED_OFFICIAL_TEST_PREDICTIONS')
display(alignment)

## 4. Identical metric implementation for both model families


In [ ]:
def probability_columns(classes): return [f'probability_{c}' for c in classes]
def evaluate(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    p=q[probability_columns(classes)].to_numpy(float)
    out={'n':len(q),'accuracy':accuracy_score(y,yp),'balanced_accuracy':balanced_accuracy_score(y,yp),
         'macro_f1':f1_score(y,yp,average='macro'),'mcc':matthews_corrcoef(y,yp),
         'roc_auc':roc_auc_score(y,p[:,1]) if len(classes)==2 else roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))}
    if len(classes)==2:
        tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel(); out.update({'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp)})
    return out

rows=[]
for (task,method),q in combined.groupby(['task','comparison_method'],sort=False): rows.append({'task':task,'method':method,**evaluate(q,task)})
metrics=pd.DataFrame(rows)
save_table(metrics,'04_IDENTICALLY_COMPUTED_MATCHED_METRICS')
display(metrics.sort_values(['task','balanced_accuracy'],ascending=[True,False]).round(4))

## 5. Paired recording-level bootstrap comparison

Within each endpoint and bootstrap repetition, the same class-stratified recording indices are applied to both methods. The primary contrast is

\[
\Delta_{BA}=BA_{\mathrm{selected}}-BA_{\mathrm{conventional}}.
\]

Thus, negative values favor the conventional benchmark. Two-sided bootstrap sign probabilities are Holm-adjusted across the four endpoint-level primary comparisons.

In [ ]:
rng=np.random.default_rng(SEED); comparison=[]
for task,classes in TASKS.items():
    a=conv.loc[conv.task.eq(task)].set_index('recording_id').sort_index()
    b=selected.loc[selected.task.eq(task)].set_index('recording_id').sort_index()
    strata={c:np.flatnonzero(a.true_label.to_numpy()==c) for c in classes}; deltas=[]
    for _ in range(N_BOOT):
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        qa=a.iloc[ix].reset_index(); qb=b.iloc[ix].reset_index()
        deltas.append(evaluate(qb,task)['balanced_accuracy']-evaluate(qa,task)['balanced_accuracy'])
    deltas=np.asarray(deltas)
    observed=(metrics.loc[(metrics.task.eq(task))&(metrics.method.eq(METHOD_SELECTED)),'balanced_accuracy'].iloc[0]-
              metrics.loc[(metrics.task.eq(task))&(metrics.method.eq(METHOD_CONVENTIONAL)),'balanced_accuracy'].iloc[0])
    p=2*min((np.sum(deltas<=0)+1)/(N_BOOT+1),(np.sum(deltas>=0)+1)/(N_BOOT+1)); p=min(1.0,p)
    comparison.append({'task':task,'selected_minus_conventional_ba':observed,'ci_low':np.quantile(deltas,.025),
                       'ci_high':np.quantile(deltas,.975),'unadjusted_p_value':p,'bootstrap_repetitions':N_BOOT})
comparison=pd.DataFrame(comparison); comparison['holm_adjusted_p_value']=holm_adjust(comparison.unadjusted_p_value)
comparison['interval_excludes_zero']=(comparison.ci_low>0)|(comparison.ci_high<0)
save_table(comparison,'05_PAIRED_BALANCED_ACCURACY_DIFFERENCES'); display(comparison.round(4))

## 6. Evidence reconciliation and interpretation lock


In [ ]:
decision=[]
for task in TASKS:
    q=metrics.loc[metrics.task.eq(task)].set_index('method'); d=comparison.set_index('task').loc[task]
    conv_ba=q.loc[METHOD_CONVENTIONAL,'balanced_accuracy']; sel_ba=q.loc[METHOD_SELECTED,'balanced_accuracy']
    if np.isclose(conv_ba,sel_ba): higher='Tie in balanced accuracy'
    elif conv_ba>sel_ba: higher='Frozen conventional benchmark'
    else: higher='Endpoint-selected CEEDNet/SSL'
    decision.append({'task':task,'conventional_ba':conv_ba,'selected_model_ba':sel_ba,'higher_test_point_estimate':higher,
                     'paired_difference_selected_minus_conventional':d.selected_minus_conventional_ba,
                     'paired_ci_low':d.ci_low,'paired_ci_high':d.ci_high,'holm_adjusted_p_value':d.holm_adjusted_p_value,
                     'selection_consequence':'No post-test model switching; report both frozen evidence streams',
                     'historical_85_percent_status':'Not part of this matched lock unless tied to an authenticated compatible prediction artifact'})
decision=pd.DataFrame(decision)
save_table(decision,'06_FINAL_COMPARATOR_RECONCILIATION'); display(decision.round(4))

## 7. Publication-ready comparison figures


In [ ]:
labels={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD','normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
order=list(TASKS); x=np.arange(len(order)); width=.35
conv_ba=metrics.loc[metrics.method.eq(METHOD_CONVENTIONAL)].set_index('task').loc[order,'balanced_accuracy'].to_numpy()
sel_ba=metrics.loc[metrics.method.eq(METHOD_SELECTED)].set_index('task').loc[order,'balanced_accuracy'].to_numpy()
fig,ax=plt.subplots(figsize=(10,5.5)); ax.bar(x-width/2,conv_ba,width,label='Frozen conventional',color='#4c78a8'); ax.bar(x+width/2,sel_ba,width,label='Selected CEEDNet/SSL',color='#f28e2b')
for i,task in enumerate(order): ax.hlines(1/len(TASKS[task]),i-.42,i+.42,color='#b22222',linestyle='--',linewidth=1)
ax.set_xticks(x); ax.set_xticklabels([labels[t] for t in order]); ax.set_ylabel('Balanced accuracy'); ax.set_ylim(.25,1.0); ax.legend(); ax.grid(axis='y',alpha=.25)
ax.set_title('Matched retrospective official-test comparison',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7D2_Matched_Official_Test_Comparison.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7D2_Matched_Official_Test_Comparison.png',dpi=400,bbox_inches='tight'); plt.show()

q=comparison.set_index('task').loc[order]; y=q.selected_minus_conventional_ba.to_numpy(); lo=y-q.ci_low.to_numpy(); hi=q.ci_high.to_numpy()-y
fig,ax=plt.subplots(figsize=(9,5)); ax.errorbar(np.arange(len(order)),y,yerr=[lo,hi],fmt='o',capsize=5,color='#1f4e79'); ax.axhline(0,color='#333333',linestyle='--')
ax.set_xticks(np.arange(len(order))); ax.set_xticklabels([labels[t] for t in order]); ax.set_ylabel('Selected minus conventional balanced accuracy'); ax.grid(axis='y',alpha=.25)
ax.set_title('Paired recording-level differences with 95% bootstrap intervals',fontweight='bold'); fig.tight_layout()
fig.savefig(FIGURES/'Figure_P1_E16_7D2_Paired_BA_Differences.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7D2_Paired_BA_Differences.png',dpi=400,bbox_inches='tight'); plt.show()

## 8. Final integrity and provenance lock


In [ ]:
checks=[]
for task,classes in TASKS.items():
    a=conv.loc[conv.task.eq(task)].sort_values('recording_id'); b=selected.loc[selected.task.eq(task)].sort_values('recording_id')
    for name,q in [(METHOD_CONVENTIONAL,a),(METHOD_SELECTED,b)]:
        p=q[probability_columns(classes)].to_numpy(float)
        checks.extend([
            {'scope':task,'check':f'{name}: one prediction per recording','pass':len(q)>0 and not q.recording_id.duplicated().any()},
            {'scope':task,'check':f'{name}: probabilities valid','pass':np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-6)},])
    checks.extend([
        {'scope':task,'check':'same recording IDs and order','pass':a.recording_id.reset_index(drop=True).equals(b.recording_id.reset_index(drop=True))},
        {'scope':task,'check':'same true labels','pass':a.true_label.reset_index(drop=True).equals(b.true_label.reset_index(drop=True))},
        {'scope':task,'check':'correct chance reference','pass':np.isclose(1/len(classes),.5 if len(classes)==2 else 1/3)},])
checks.extend([
    {'scope':'all','check':'four endpoints compared','pass':set(metrics.task)==set(TASKS)},
    {'scope':'all','check':'both frozen model families represented','pass':set(metrics.method)=={METHOD_CONVENTIONAL,METHOD_SELECTED}},
    {'scope':'all','check':'paired bootstrap completed','pass':len(comparison)==4 and comparison.holm_adjusted_p_value.notna().all()},
    {'scope':'all','check':'no EEG or official-test signal reloaded','pass':True},
    {'scope':'all','check':'no refitting, threshold change, or recalibration','pass':True},
    {'scope':'all','check':'no external dataset accessed','pass':True},
    {'scope':'all','check':'historically untouched claim prohibited','pass':True},])
checks=pd.DataFrame(checks); save_table(checks,'07_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7D-2 integrity failure.')

run={'experiment':'P1-E16.7D-2 matched official-test comparison of frozen conventional and selected models','status':'PASSED',
     'completed_utc':datetime.now(timezone.utc).isoformat(),'endpoints':TASKS,'metrics':metrics.to_dict('records'),
     'paired_comparisons':comparison.to_dict('records'),'evidence_label':'retrospective matched locked-test comparison',
     'models_retrained':False,'test_signals_reloaded':False,'thresholds_changed':False,'external_data_accessed':False,
     'historically_untouched_claim':False,'integrity_passed':True,'input_files':inputs.to_dict('records')}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7D-2 PASSED — MATCHED CONVENTIONAL VERSUS SELECTED-MODEL OFFICIAL-TEST COMPARISON COMPLETE')